# Prueba de modelos generados

In [1]:
# Importamos librerias 
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import requests
from tqdm import tqdm
import joblib
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score


In [2]:
# ================== CONFIGURACIÓN ==================
BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"
FILES = [
    "P_DEMO.xpt",      # Demografía (edad, sexo, etc.)
    "P_CBC.xpt",       # Hemograma completo (plaquetas, leucocitos, glóbulos rojos, etc.)
    "P_BIOPRO.xpt",    # Bioquímica (ALT, AST, Creatinina, Glucosa, Colesterol total, etc.)
    "P_TCHOL.xpt",     # Colesterol total
    "P_HDL.xpt",       # HDL
    "P_TRIGLY.xpt",    # Triglicéridos + LDL
    "P_GHB.xpt",       # HbA1c
    "P_INS.xpt",       # Insulina
    "P_HSCRP.xpt",     # Proteína C Reactiva
    "P_GLU.xpt",       # Glucosa en ayunas
    "P_BMX.xpt",       # BMI, peso, talla
    "P_BPXO.xpt",      # Presión arterial + frecuencia cardíaca
]

OUTPUT_FOLDER = "data_test"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ================== DESCARGA ==================
def download_file(filename):
    url = BASE_URL + filename
    filepath = os.path.join(OUTPUT_FOLDER, filename)
    if os.path.exists(filepath):
        print(f"{filename} ya existe")
        return filepath
    print(f"Descargando {filename}...")
    response = requests.get(url, stream=True)
    with open(filepath, "wb") as f:
        for chunk in tqdm(response.iter_content(chunk_size=8192), desc=filename):
            f.write(chunk)
    return filepath

print("Iniciando descarga de archivos NHANES...\n")
file_paths = [download_file(f) for f in FILES]

# ================== LECTURA Y UNIÓN ==================
print("\nLeyendo y uniendo archivos...")

dfs = {}
for filepath in file_paths:
    filename = os.path.basename(filepath)
    df = pd.read_sas(filepath, format='xport', encoding='utf-8')
    dfs[filename] = df
    print(f"✓ {filename} → {df.shape[0]:,} filas, {df.shape[1]} columnas")

# Unir todo por SEQN
merged = dfs["P_DEMO.xpt"].copy()

for name, df in dfs.items():
    if name != "P_DEMO.xpt":
        merged = pd.merge(merged, df, on='SEQN', how='inner')

print(f"\nDataFrame final: {merged.shape[0]:,} pacientes × {merged.shape[1]} columnas")

# Guardar resultados
merged.to_csv(os.path.join(OUTPUT_FOLDER, "NHANES_2017-2020_completo.csv"), index=False)
merged.to_pickle(os.path.join(OUTPUT_FOLDER, "NHANES_2017-2020_completo.pkl"))

print("\n¡Todo listo!")
print(f"Archivos guardados en la carpeta: ./{OUTPUT_FOLDER}/")
print("   - NHANES_2017-2020_completo.csv  (para Excel / abrir fácilmente)")
print("   - NHANES_2017-2020_completo.pkl  (más rápido para Python)")

Iniciando descarga de archivos NHANES...

Descargando P_DEMO.xpt...


P_DEMO.xpt: 74it [00:00, 356.64it/s]


Descargando P_CBC.xpt...


P_CBC.xpt: 71it [00:00, 210.67it/s]


Descargando P_BIOPRO.xpt...


...

Descargando P_BMX.xpt...


P_BMX.xpt: 49it [00:00, 219.87it/s]


Descargando P_BPXO.xpt...


P_BPXO.xpt: 26it [00:00, 430.07it/s]



Leyendo y uniendo archivos...
✓ P_DEMO.xpt → 15,560 filas, 29 columnas
✓ P_CBC.xpt → 13,772 filas, 22 columnas
✓ P_BIOPRO.xpt → 10,409 filas, 41 columnas
✓ P_TCHOL.xpt → 12,198 filas, 3 columnas
✓ P_HDL.xpt → 12,198 filas, 3 columnas
✓ P_TRIGLY.xpt → 5,090 filas, 10 columnas
✓ P_GHB.xpt → 10,409 filas, 2 columnas
✓ P_INS.xpt → 5,090 filas, 5 columnas
✓ P_HSCRP.xpt → 13,772 filas, 3 columnas
✓ P_GLU.xpt → 5,090 filas, 4 columnas
✓ P_BMX.xpt → 14,300 filas, 22 columnas
✓ P_BPXO.xpt → 11,656 filas, 12 columnas

DataFrame final: 5,090 pacientes × 145 columnas

¡Todo listo!
Archivos guardados en la carpeta: ./data_test/
   - NHANES_2017-2020_completo.csv  (para Excel / abrir fácilmente)
   - NHANES_2017-2020_completo.pkl  (más rápido para Python)


In [ ]:
#Ahora tenemos que seleccionar solo los datos que necesitamos para evaluar el modelo
# Para esto, vamos a seleccionar las siguientes columnas:
features = [
    'Platelets',
    'White Blood Cells',
    'Red Blood Cells',
    'Hematocrit',
    'Mean Corpuscular Volume',
    'Mean Corpuscular Hemoglobin',
    'Mean Corpuscular Hemoglobin Concentration',
    'HDL Cholesterol',
    'ALT',
    'Heart Rate'
]  

# Equivalencias en codigos NHANES:
equivalencias = {
    'Platelets': 'P_CBC.xpt',
    'White Blood Cells': 'P_CBC.xpt',
    'Red Blood Cells': 'P_CBC.xpt',
    'Hematocrit': 'P_CBC.xpt',
    'Mean Corpuscular Volume': 'P_CBC.xpt',
    'Mean Corpuscular Hemoglobin': 'P_CBC.xpt',
    'Mean Corpuscular Hemoglobin Concentration': 'P_CBC.xpt',
    'HDL Cholesterol': 'P_HDL.xpt',
    'ALT': 'P_BIOPRO.xpt',
    'Heart Rate': 'P_BPXO.xpt'
}


NHANES_completo = pd.read_csv("NHANES_2017-2020_completo.csv")


